In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import h5py
import skimage as sk
from sklearn import mixture
import pandas as pd

import dask.dataframe as dd
import matplotlib.patches as patches
from dask.distributed import wait

import xarray as xr
import matplotlib.colors as mcolors

colors = list(mcolors.TABLEAU_COLORS.values())
%matplotlib inline

# import paulssonlab.paulssonlab.src.paulssonlab.deaton.trenchripper.trenchripper as tr

import paulssonlab.deaton.trenchripper.trenchripper as tr

import warnings
warnings.filterwarnings(action="once")

import matplotlib
matplotlib.rcParams["figure.figsize"] = [20, 10]

In [ ]:
df_cells = pd.read_csv('/home/lag36/scratch/lag36/2022-12-21_lDE25_Run_1/Growth_Division/TEST_DFS_NUCS/cells_part0.csv', index_col='File Parquet Index')

In [ ]:
dfs = []
for i in range(4):
    dfs.append( pd.read_csv('/home/lag36/scratch/lag36/2022-12-21_lDE25_Run_1/Growth_Division/TEST_DFS_NUCS/nucs_part'+str(i)+'.csv', index_col='Cell Parquet Index') )
df_nucs = pd.concat(dfs)
df_nucs = df_nucs.drop(-1)

In [ ]:
def plot_function(dfy_allt, dfm_allt, timepoint, channel = 'mCherry', use_colors=True, no_yfp = False):
    
    if not no_yfp:
        dfy_single = dfy_allt[dfy_allt["timepoints"] == timepoint]
        dfy_single_indices = dfy_single.index.to_numpy()
    dfm_single = dfm_allt[dfm_allt["timepoints"] == timepoint]
    
    # print(dfy_single.columns)
    channels = [channel]
    t_inds = [timepoint]
    with h5py.File("/home/lag36/scratch/lag36/2022-12-21_lDE25_Run_1/Growth_Division/kymograph/kymograph_" + str(file_idx) + ".hdf5", "r") as infile:  
        for ch in channels:
            img = infile[ch][idx][t_inds[0]]
            if len(t_inds) > 1:
                for t_ind in t_inds[1:]:
                    img_ind = infile[ch][idx][t_ind]
                    img = np.concatenate((img, img_ind), axis = 1)

    fig, ax = plt.subplots();
    ax.imshow(img, cmap='gray');
    
    if not no_yfp:
        for i in range(len(dfy_single)):
            row = dfy_single.iloc[i]
            # ax.plot(row['Centroid X']/pixel_microns, row['Centroid Y']/pixel_microns, 'o')

            n_width = row['bbox_max_col']-row['bbox_min_col']
            n_height = row['bbox_max_row']-row['bbox_min_row']

            # SHOW CELL EDGE WITH COLOR
            ax.plot(row['bbox_min_col']/pixel_microns, row['bbox_min_row']/pixel_microns, 'o', markersize = 4, color = colors[dfy_single_indices[i]%len(colors)]);
            
            rect = patches.Rectangle((row['bbox_min_col']/pixel_microns, row['bbox_min_row']/pixel_microns), 
                                     n_width/pixel_microns,
                                     n_height/pixel_microns,
                                     facecolor = 'none', edgecolor='w', linewidth = 1);
            ax.add_patch(rect);
    
    # SHOW NUCLEOID CENTROID
    for i in range(len(dfm_single)):
        if use_colors:
            
            plt.plot(dfm_single.iloc[i]['centroid_x']/pixel_microns, dfm_single.iloc[i]['centroid_y']/pixel_microns, 'o', markersize = 4, 
                     color = colors[dfm_single.reset_index()["Cell Parquet Index"].iloc[i]%len(colors)]);
        else:
            plt.plot(dfm_single.iloc[i]['centroid_x']/pixel_microns, dfm_single.iloc[i]['centroid_y']/pixel_microns, 'o', markersize = 4, color = 'r');
            

In [ ]:
pixel_microns = 0.105951961895293
kymo_length_y = 290*pixel_microns # CHANGE THIS
kymo_width_x = 31*pixel_microns # CHANGE THIS

In [ ]:
# How do I quickly
# file_idx = 5
# idx = 6
# timepoint = 163 
file_idx = 0
idx = 0
timepoint = 120

test_nuc = df_nucs[df_nucs['File Index'] == file_idx]
test_lineage = df_cells[df_cells['File Index'] == file_idx]

# plot_function(dfy_allt=test_lineage, dfm_allt=test_nuc, timepoint=0, use_colors=False)
plot_function(dfy_allt=test_lineage[test_lineage['File Trench Index']==idx], 
              dfm_allt=test_nuc[test_nuc["File Trench Index"]==idx], 
              timepoint=timepoint, 
              channel = 'mCherry',
              use_colors=True, 
              no_yfp=False);

In [ ]:
file_idx = 0
idx = 0
timepoint = 120

test_nuc = df_nucs[df_nucs['File Index'] == file_idx]
test_lineage = df_cells[df_cells['File Index'] == file_idx]

test_nuc = test_nuc[test_nuc['File Trench Index'] == idx]
test_lineage = test_lineage[test_lineage['File Trench Index'] == idx]

test_nuc = test_nuc[test_nuc['timepoints'] == timepoint]
test_lineage = test_lineage[test_lineage['timepoints'] == timepoint]
                                        

In [ ]:
test_nuc

## Test the mean area by nucleoid

In [ ]:
nuc_mean_area = test_nuc['area'].groupby('Cell Parquet Index').mean()
nuc_std_area = test_nuc['area'].groupby('Cell Parquet Index').std()

## Test the sorted nucleoid distance

In [ ]:
def get_mean_and_groupbyinternuc_distance():

In [ ]:
dfs_cells_mean = test_nuc['centroid_y'].groupby('Cell Parquet Index').apply(lambda x : np.mean(np.diff(x.sort_values())))
dfs_cells_std = test_nuc['centroid_y'].groupby('Cell Parquet Index').apply(lambda x : np.std(np.diff(x.sort_values())))
dfs_cells_std
dfs_cells_cv = dfs_cells_std/dfs_cells
dfs_cells_cv.replace({0:np.nan})

## Test the pairwise distance

In [ ]:
a = test_nuc[['centroid_x', 'centroid_y']].groupby('Cell Parquet Index')

In [ ]:
def get_nucleoid_distance_matrix(centroids):
    centroids = centroids.to_numpy()
    centroids_diff_sq = np.square(centroids[:,None,:] - centroids[None,:,:])
    distance_matrix = np.sqrt(np.sum(centroids_diff_sq, axis=2)) # This is the distance matrix (n_nuc x n_nuc)
    distances = distance_matrix[np.triu_indices(distance_matrix.shape[0], k = 1)] # This is the list of the upper triangle of the matrix (all pairwise distances without repetition)
    
    return distances

In [ ]:
a.indices

In [ ]:
get_nucleoid_distance_matrix(a.get_group(1200071))

In [ ]:
g1 = a.get_group(1200075)

In [ ]:
g1

In [ ]:
g1n = g1.to_numpy()

In [ ]:
g1d= g1n[:,None,:] - g1n[None,:,:]

In [ ]:
g1d = np.square(g1d)

In [ ]:
g1d = np.sqrt(np.sum(g1d, axis=2))

In [ ]:
g1d

In [ ]:
np.triu(g1d)

In [ ]:
np.triu_indices(g1d.shape[0])

In [ ]:
g1d[np.triu_indices(g1d.shape[0], k = 1)]